In [0]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/123/netflix_titles.csv")


In [0]:
df.groupBy("type").count().show()

+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2676|
|        Movie| 6131|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



In [0]:
from pyspark.sql.functions import col

null_D = df.filter(col("director").isNull() | (col("director") == "")).count()
print(f"total of null directors: {null_D}")

total of null directors: 2636


In [0]:
df.groupBy("rating").count().orderBy(col("count").desc()).show()

+------------+-----+
|      rating|count|
+------------+-----+
|       TV-MA| 3195|
|       TV-14| 2158|
|       TV-PG|  862|
|           R|  796|
|       PG-13|  489|
|       TV-Y7|  334|
|        TV-Y|  307|
|          PG|  286|
|        TV-G|  220|
|          NR|   80|
|           G|   41|
|        NULL|    6|
|    TV-Y7-FV|    6|
|          UR|    3|
|       NC-17|    3|
|        2021|    2|
|        2019|    1|
|        2017|    1|
| Jide Kosoko|    1|
|        2006|    1|
+------------+-----+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col

df.groupBy("listed_in").count().orderBy(col("count").desc()).show(15, truncate=False)

+------------------------------------------------+-----+
|listed_in                                       |count|
+------------------------------------------------+-----+
|Dramas, International Movies                    |361  |
|Documentaries                                   |358  |
|Stand-Up Comedy                                 |334  |
|Comedies, Dramas, International Movies          |273  |
|Dramas, Independent Movies, International Movies|252  |
|Kids' TV                                        |220  |
|Children & Family Movies                        |215  |
|Children & Family Movies, Comedies              |201  |
|Documentaries, International Movies             |186  |
|Dramas, International Movies, Romantic Movies   |180  |
|Comedies, International Movies                  |176  |
|Comedies, International Movies, Romantic Movies |152  |
|Dramas                                          |138  |
|Dramas, International Movies, Thrillers         |134  |
|Action & Adventure, Dramas, In

In [0]:
from pyspark.sql.functions import try_to_date, month, year, trim, col

df_dates = df.withColumn("parsed_date", try_to_date(trim(col("date_added")), "MMMM d, yyyy"))
df_dates.filter(col("parsed_date").isNotNull()) \
        .groupBy(year("parsed_date").alias("year"), month("parsed_date").alias("month")) \
        .count() \
        .orderBy(col("year").desc(), col("month").desc()) \
        .show(15)

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2021|    9|  181|
|2021|    8|  177|
|2021|    7|  257|
|2021|    6|  206|
|2021|    5|  132|
|2021|    4|  188|
|2021|    3|  111|
|2021|    2|  109|
|2021|    1|  130|
|2020|   12|  168|
|2020|   11|  153|
|2020|   10|  167|
|2020|    9|  168|
|2020|    8|  128|
|2020|    7|  145|
+----+-----+-----+
only showing top 15 rows


In [0]:
from pyspark.sql.functions import regexp_extract, avg, round, col

df.filter((col("type") == "TV Show") & col("duration").rlike(r"\d+")) \
  .withColumn("seasons_num", regexp_extract(col("duration"), r"(\d+)", 1).cast("int")) \
  .agg(round(avg("seasons_num"), 2).alias("avg_seasons")) \
  .show()

+-----------+
|avg_seasons|
+-----------+
|       1.77|
+-----------+



In [0]:

#
from pyspark.sql.functions import col

horror_latest_10 = df.filter(col("listed_in").contains("Horror")) \
                     .orderBy(col("release_year").desc())

horror_latest_10.select("title", "release_year", "listed_in").show(10, truncate=False)

+---------------------------+------------+-------------------------------------------------------+
|title                      |release_year|listed_in                                              |
+---------------------------+------------+-------------------------------------------------------+
|Aftermath                  |2021        |Horror Movies                                          |
|Boomika (Malayalam)        |2021        |Horror Movies, International Movies, Thrillers         |
|Midnight Mass              |2021        |TV Dramas, TV Horror, TV Mysteries                     |
|Boomika                    |2021        |Horror Movies, International Movies, Thrillers         |
|Brand New Cherry Flavor    |2021        |TV Dramas, TV Horror, TV Mysteries                     |
|Boomika (Telugu)           |2021        |Horror Movies, International Movies, Thrillers         |
|Blood Red Sky              |2021        |Action & Adventure, Horror Movies, International Movies|
|Kingdom: 

In [0]:
from pyspark.sql.functions import split, trim, col, try_element_at, lit

df_cast_split = df.withColumn("cast_array", split(col("cast"), ","))

# Envolvemos los índices 1 y 2 con lit()
ryan_movies = df_cast_split.filter(
    (trim(try_element_at(col("cast_array"), lit(1))) == "Ryan Reynolds") | 
    (trim(try_element_at(col("cast_array"), lit(2))) == "Ryan Reynolds")
)

ryan_top_5 = ryan_movies.orderBy(col("release_year").desc())
ryan_top_5.select("title", "release_year", "cast").show(5, truncate=False)

+-----------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title            |release_year|cast                                                                                                                                                                    |
+-----------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|6 Underground    |2019        |Ryan Reynolds, Mélanie Laurent, Corey Hawkins, Dave Franco, Adria Arjona, Manuel Garcia-Rulfo, Ben Hardy, Lior Raz, Payman Maadi, Yuri Kolokolnikov, Kim Kold           |
|Mississippi Grind|2015        |Ryan Reynolds, Ben Mendelsohn, Sienna Miller, Analeigh Tipton, Alfre Woodard, James Toback, Robin Weigert                                                       

In [0]:
from pyspark.sql.functions import try_to_date, trim, col

# Cambiamos to_date por try_to_date y agregamos trim para limpiar espacios
df_dates = df.withColumn("parsed_date", try_to_date(trim(col("date_added")), "MMMM d, yyyy"))

covid_season_df = df_dates.filter(
    (col("parsed_date") >= "2020-06-01") & 
    (col("parsed_date") <= "2023-05-31")
)

print(f"Total de contenidos lanzados en temporada COVID: {covid_season_df.count()}")
covid_season_df.select("title", "date_added", "type").show(10, truncate=False)

Total de contenidos lanzados en temporada COVID: 2576
+--------------------------------+------------------+-------+
|title                           |date_added        |type   |
+--------------------------------+------------------+-------+
|Dick Johnson Is Dead            |September 25, 2021|Movie  |
|Blood & Water                   |September 24, 2021|TV Show|
|Ganglands                       |September 24, 2021|TV Show|
|Jailbirds New Orleans           |September 24, 2021|TV Show|
|Kota Factory                    |September 24, 2021|TV Show|
|Midnight Mass                   |September 24, 2021|TV Show|
|My Little Pony: A New Generation|September 24, 2021|Movie  |
|Sankofa                         |September 24, 2021|Movie  |
|The Great British Baking Show   |September 24, 2021|TV Show|
|The Starling                    |September 24, 2021|Movie  |
+--------------------------------+------------------+-------+
only showing top 10 rows


In [0]:
blood_red_sky = df.filter(col("title") == "Blood Red Sky")
blood_red_sky.show(truncate=False)

+-------+-----+-------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-------+-------------+------------+------+--------+-------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|show_id|type |title        |director       |cast                                                                                                                                                 |country|date_added   |release_year|rating|duration|listed_in                                              |description                                                                                                                                             |
+-------+-----+-------------+---------------+-------------------------------------------